# Big Data Analysis — Group Project
## Notebook 1: Setup, Data Ingestion, ETL, Cleaning & Exploration

**Client scenario**: we're acting as consultants for an imaginary e-commerce client that
sells food products online. The client has hundreds of thousands of customer reviews
and wants to know:

1. What makes a review *helpful* to other customers.
2. How customers feel about their products (sentiment).
3. Which products and users are central to their review network (covered in the GraphFrames notebook later).

**Dataset**: [Amazon Fine Food Reviews](https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews)
— around 568k reviews from 1999 to 2012. This isn't true "big data" in size, but the brief
explicitly says that's fine. What matters is that our code would still work if the data
were much bigger. We flag anywhere we have to break that rule.

**This notebook covers**:
- Spark setup
- Loading the raw data as an RDD (to show we understand the basics)
- Re-loading it as a DataFrame with a proper schema
- Cleaning: fixing types, deriving columns, handling nulls and duplicates
- Exploring the data with DataFrames and Spark SQL
- Saving the cleaned data as Parquet for the next notebooks

---
## 1. Environment Setup

We install PySpark and Java 17 (Spark runs on the JVM). Only needed once per session
— Colab resets the environment each time.

In [1]:
# Install PySpark and plotting libraries
!pip install pyspark plotly "pandas>=2.2.0" "nbformat>=4.2.0" --quiet

In [2]:
# Install Java 17 — required for Spark
!sudo apt-get update -qq
!sudo apt-get install -y openjdk-17-jdk-headless -qq
!java -version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 2.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
(Reading database ... 118242 files and directories currently installed.)
Preparing to unpack .../openjdk-17-jdk-headless_17.0.19+10-1~22.04.2_amd64.deb ...
Unpacking openjdk-17-jdk-headless:amd64 (17.0.19+10-1~22.04.2) over (17.0.18+8-1~22.04.1) ...
Preparing to unpack .../openjdk-17-jre-headless_17.0.19+10-1~22.04.2_amd64.deb ...
Unpacking openjdk-17-jre-head

In [3]:
# Tell PySpark where Java lives
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

### 1.1 Creating the Spark Session

The `SparkSession` is our entry point to everything Spark can do (DataFrames, SQL,
Streaming, MLlib). A few config choices worth noting:

- **`master("local[*]")`** — run locally using all available cores. On a real cluster
  this would point to YARN or Kubernetes.
- **memory settings** — give the JVM enough heap so it doesn't spill to disk too often.
- **`shuffle.partitions = 16`** — Spark's default is 200, which is way too many for a
  single laptop. On a real cluster you'd set this based on cluster size.
- **Adaptive Query Execution** — lets Spark re-plan queries at runtime based on the actual
  data, instead of sticking with the original plan.

In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
        .master("local[*]")
        .appName("BigDataProject_FoodReviews")
        .config("spark.executor.memory", "4g")
        .config("spark.driver.memory", "4g")
        # Lower than the default 200 — better fit for a single machine
        .config("spark.sql.shuffle.partitions", "16")
        # Adaptive Query Execution — Spark optimises plans at runtime
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
        .getOrCreate()
)

# Less log spam
spark.sparkContext.setLogLevel("WARN")

# SparkContext is still needed for RDD work below
sc = spark.sparkContext

print(f"Spark version : {spark.version}")
print(f"Master        : {sc.master}")
print(f"App name      : {sc.appName}")
print(f"Default # of partitions : {sc.defaultParallelism}")

Spark version : 4.0.2
Master        : local[*]
App name      : BigDataProject_FoodReviews
Default # of partitions : 2


In [5]:
# Mount Google Drive so we can read the CSV and persist outputs across sessions.
# Colab wipes its local disk when the session ends — Drive doesn't.
# This matters because Notebook 2 needs to read the Parquet we write here.
from google.colab import drive
drive.mount('/content/drive')

# Project folder — everything (inputs, outputs) lives here
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks"

Mounted at /content/drive


---
## 2. Loading the Data as an RDD

Before using the high-level DataFrame API, we load the file as an RDD. Two reasons:

1. The brief explicitly asks for RDD usage.
2. It shows we understand what DataFrames are built on top of.

We use `sc.textFile`, which reads the file line by line and splits it across partitions
automatically. Nothing actually happens yet — Spark only executes when we call an action.

> **Note**: download `Reviews.csv` from Kaggle and put it in your Drive project folder.

In [6]:
# Path to the raw CSV in Drive
DATA_PATH = f"{PROJECT_DIR}/Reviews.csv"

# Load as an RDD of strings (one element per line). This is lazy.
reviews_raw_rdd = sc.textFile(DATA_PATH)

# Quick peek — take(2) only reads enough partitions to satisfy the request
for line in reviews_raw_rdd.take(2):
    print(line[:200], "...\n")

Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text ...

1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The product  ...



### 2.1 Counting records and partitions

In [7]:
# Full count — triggers a full scan, but useful as a sanity check
n_lines = reviews_raw_rdd.count()
print(f"Total lines in file (incl. header): {n_lines:,}")

# Number of partitions Spark created based on file size
print(f"Number of partitions: {reviews_raw_rdd.getNumPartitions()}")

Total lines in file (incl. header): 568,455
Number of partitions: 9


### 2.2 Removing the header and a quick Map-Reduce demo

To show the classic RDD operations (`filter`, `flatMap`, `map`, `reduceByKey`), we run a
word-count on the review summaries.

One thing worth pointing out: we use `top()` at the end, not `sortBy().take()`. `top()`
keeps a small heap per partition and combines them — much cheaper than a global sort
across the cluster.

In [8]:
import csv

def parse_csv_line(line):
    """
    Parse a CSV line into a list of fields. Using the csv module (not a naive split)
    matters because review text contains commas and quotes.
    """
    reader = csv.reader([line])
    return next(reader, [])

# Get the header so we can filter it out
header = reviews_raw_rdd.first()
print(f"Header: {header}")

# Filter out the header line, then parse the CSV. Both are narrow transformations
# (no shuffle).
reviews_rdd = reviews_raw_rdd.filter(lambda row: row != header).map(parse_csv_line)

# Drop malformed rows that don't have all 10 fields
reviews_rdd = reviews_rdd.filter(lambda fields: len(fields) == 10)

print(f"Number of well-formed review records (RDD): {reviews_rdd.count():,}")

Header: Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
Number of well-formed review records (RDD): 568,454


### 2.3 Caching

We'll run several actions on `reviews_rdd` below. Without caching, Spark would re-parse
the CSV from scratch every time. `cache()` keeps it in memory after the first action.

This is one of the most important performance tools in Spark.

In [9]:
import time

# Mark the RDD for caching — actual caching happens after the first action
reviews_rdd.cache()

# First pass — reads from disk AND fills the cache
t0 = time.time()
n1 = reviews_rdd.count()
print(f"First count (cold cache):   {n1:,} records  in {time.time() - t0:.2f}s")

# Second pass — reads from memory, much faster
t0 = time.time()
n2 = reviews_rdd.count()
print(f"Second count (warm cache):  {n2:,} records  in {time.time() - t0:.2f}s")

First count (cold cache):   568,454 records  in 15.55s
Second count (warm cache):  568,454 records  in 2.74s


### 2.4 Word-count on review summaries

The classic Map-Reduce pattern: split each summary into words, emit `(word, 1)` pairs,
then sum.

A small but important point: we use `reduceByKey`, not `groupByKey`. `reduceByKey`
combines values locally on each partition *before* shuffling, which means much less
data moves across the network. `groupByKey` would send every individual value across
the network — dangerous on big datasets.

In [10]:
import string

# Index of the "Summary" field in our parsed rows
SUMMARY_IDX = 8

# Quick stopword list for this demo. The ML notebook uses Spark's StopWordsRemover
# instead — keeping things consistent.
STOPWORDS = {
    "the", "a", "an", "and", "or", "but", "is", "are", "was", "were", "be",
    "this", "that", "these", "those", "i", "you", "he", "she", "it", "we", "they",
    "to", "of", "in", "on", "for", "with", "as", "at", "by", "from", "my", "your",
    "have", "has", "had", "do", "does", "did", "not", "no", "so", "if", "than",
    "very", "just", "too", "also",
}

def tokenize(text):
    """Lowercase, strip punctuation, split on whitespace, drop stopwords."""
    text = text.lower().translate(str.maketrans("", "", string.punctuation))
    return [w for w in text.split() if w and w not in STOPWORDS]

# Full Map-Reduce pipeline:
#   map      -> grab the Summary field
#   flatMap  -> split into words (one row becomes many)
#   map      -> emit (word, 1) pairs
#   reduceByKey -> sum the counts (combined locally first to minimise shuffle)
summary_word_counts = (
    reviews_rdd
        .map(lambda fields: fields[SUMMARY_IDX])
        .flatMap(tokenize)
        .map(lambda word: (word, 1))
        .reduceByKey(lambda a, b: a + b)
)

# top() is much cheaper than sortBy + take — no global sort needed
top_summary_words = summary_word_counts.top(15, key=lambda kv: kv[1])

print("Top 15 words in review summaries:")
for word, count in top_summary_words:
    print(f"  {word:<15} {count:>8,}")

Top 15 words in review summaries:
  great             72,612
  good              51,494
  best              33,367
  love              27,414
  coffee            24,785
  tea               21,889
  product           19,766
  delicious         18,721
  taste             17,595
  flavor            13,541
  like              13,305
  excellent         13,178
  food              13,161
  dog               13,056
  tasty             11,006


We've shown the basics: RDD loading, transformations, lazy evaluation, caching, and
Map-Reduce. From here on we switch to the DataFrame API, which is much faster (thanks
to the Catalyst optimiser) and easier to use for structured data like ours.

In [11]:
# Free the RDD's cache — we won't need it again
reviews_rdd.unpersist()

PythonRDD[6] at RDD at PythonRDD.scala:56

---
## 3. Loading as a DataFrame with an Explicit Schema

We now load the same file as a DataFrame, but with an **explicit schema** instead of
`inferSchema=True`. Why?

| | `inferSchema=True` | Explicit schema |
|---|---|---|
| Convenience | Easy, no boilerplate | More code |
| Cost | Does an extra full pass over the data to figure out types | Single pass |
| Reliability | Can guess wrong on messy columns | Deterministic |

For big data, the extra pass `inferSchema` does is a real cost. Defining the schema
ourselves is the standard for production code.

In [12]:
from pyspark.sql.types import (
    StructType, StructField, IntegerType, LongType, StringType
)

# Define the schema explicitly — no inference pass needed
reviews_schema = StructType([
    StructField("Id",                       IntegerType(), nullable=False),
    StructField("ProductId",                StringType(),  nullable=True),
    StructField("UserId",                   StringType(),  nullable=True),
    StructField("ProfileName",              StringType(),  nullable=True),
    StructField("HelpfulnessNumerator",     IntegerType(), nullable=True),
    StructField("HelpfulnessDenominator",   IntegerType(), nullable=True),
    StructField("Score",                    IntegerType(), nullable=True),
    StructField("Time",                     LongType(),    nullable=True),  # Unix epoch seconds
    StructField("Summary",                  StringType(),  nullable=True),
    StructField("Text",                     StringType(),  nullable=True),
])

reviews_df = (
    spark.read
         .option("header", "true")
         .option("multiLine", "true")          # review text can span multiple lines
         .option("quote", '"')
         .option("escape", '"')                # CSV escapes quotes by doubling them
         .schema(reviews_schema)
         .csv(DATA_PATH)
)

print("Schema:")
reviews_df.printSchema()
print(f"\nRow count: {reviews_df.count():,}")
print(f"Partitions: {reviews_df.rdd.getNumPartitions()}")

Schema:
root
 |-- Id: integer (nullable = true)
 |-- ProductId: string (nullable = true)
 |-- UserId: string (nullable = true)
 |-- ProfileName: string (nullable = true)
 |-- HelpfulnessNumerator: integer (nullable = true)
 |-- HelpfulnessDenominator: integer (nullable = true)
 |-- Score: integer (nullable = true)
 |-- Time: long (nullable = true)
 |-- Summary: string (nullable = true)
 |-- Text: string (nullable = true)


Row count: 568,454
Partitions: 1


### 3.1 First look at the data

In [13]:
# show() truncates long strings by default — fine for a quick look
reviews_df.show(5, truncate=80)

+---+----------+--------------+-------------------------------+--------------------+----------------------+-----+----------+---------------------+--------------------------------------------------------------------------------+
| Id| ProductId|        UserId|                    ProfileName|HelpfulnessNumerator|HelpfulnessDenominator|Score|      Time|              Summary|                                                                            Text|
+---+----------+--------------+-------------------------------+--------------------+----------------------+-----+----------+---------------------+--------------------------------------------------------------------------------+
|  1|B001E4KFG0|A3SGXH7AUHU8GW|                     delmartian|                   1|                     1|    5|1303862400|Good Quality Dog Food|I have bought several of the Vitality canned dog food products and have found...|
|  2|B00813GRG4|A1D87F6ZCVE5NK|                         dll pa|                   0|    

---
## 4. ETL & Cleaning

The cleaning steps:
1. Convert the Unix `Time` column into a proper timestamp and pull out year/month.
2. Compute `HelpfulnessRatio` safely (no divide-by-zero).
3. Drop logically impossible rows (e.g. helpful votes > total votes).
4. Remove exact duplicates and "soft" duplicates.
5. Add some derived features (review length, word count) for use later.

Throughout, we use Spark's built-in functions (`F.when`, `F.from_unixtime`, etc.) rather
than Python UDFs. UDFs work, but they kill performance — they break out of the JVM and
prevent Spark from optimising the query.

In [14]:
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType

# ---------------------------------------------------------------------------
# Step 1: Convert Time to a timestamp, then derive Year and Month
# ---------------------------------------------------------------------------
reviews_clean_df = (
    reviews_df
        .withColumn("ReviewDate", F.from_unixtime(F.col("Time")).cast(TimestampType()))
        .withColumn("Year",       F.year("ReviewDate"))
        .withColumn("Month",      F.month("ReviewDate"))
)

# ---------------------------------------------------------------------------
# Step 2: Helpfulness ratio (safely)
# ---------------------------------------------------------------------------
# When the denominator is 0 (no one voted on helpfulness), we set the ratio to NULL,
# not 0. "No votes" is genuinely different information from "voted, found unhelpful".
reviews_clean_df = reviews_clean_df.withColumn(
    "HelpfulnessRatio",
    F.when(F.col("HelpfulnessDenominator") > 0,
           F.col("HelpfulnessNumerator") / F.col("HelpfulnessDenominator"))
     .otherwise(F.lit(None))
)

# ---------------------------------------------------------------------------
# Step 3: Filter out logically impossible rows
# ---------------------------------------------------------------------------
# You can't have more "helpful" votes than total votes — these are data errors.
before = reviews_clean_df.count()
reviews_clean_df = reviews_clean_df.filter(
    F.col("HelpfulnessNumerator") <= F.col("HelpfulnessDenominator")
)
after = reviews_clean_df.count()
print(f"Dropped {before - after:,} rows where Numerator > Denominator.")

# Score must be 1-5
before = reviews_clean_df.count()
reviews_clean_df = reviews_clean_df.filter(F.col("Score").between(1, 5))
after = reviews_clean_df.count()
print(f"Dropped {before - after:,} rows with invalid Score.")

Dropped 2 rows where Numerator > Denominator.
Dropped 0 rows with invalid Score.


### 4.1 Removing duplicates

This dataset has known duplicates — sometimes the same product gets re-listed under
different IDs but with identical reviews. We handle this in two passes:

- **Exact duplicates**: same in every column.
- **Soft duplicates**: same UserId + Time + Text. These are almost certainly the same
  review on duplicated products.

In [15]:
n_before = reviews_clean_df.count()

# Pass 1: drop exact duplicates
reviews_clean_df = reviews_clean_df.dropDuplicates()
n_after_exact = reviews_clean_df.count()

# Pass 2: drop soft duplicates (same user + time + text)
reviews_clean_df = reviews_clean_df.dropDuplicates(["UserId", "Time", "Text"])
n_after_soft = reviews_clean_df.count()

print(f"Rows before dedup            : {n_before:,}")
print(f"After exact-duplicate removal : {n_after_exact:,}  (-{n_before - n_after_exact:,})")
print(f"After soft-duplicate removal  : {n_after_soft:,}  (-{n_after_exact - n_after_soft:,})")

Rows before dedup            : 568,452
After exact-duplicate removal : 568,452  (-0)
After soft-duplicate removal  : 393,890  (-174,562)


### 4.2 Null check

Before adding more columns, let's see how many nulls there are. We count nulls in
every column in a single aggregation — much better than looping.

In [16]:
# Build one expression per column that counts its nulls. Single Spark job, not one
# per column.
null_counts_df = reviews_clean_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in reviews_clean_df.columns
])

# This is a 1-row result — safe to collect
null_counts_row = null_counts_df.collect()[0].asDict()

total_rows = reviews_clean_df.count()
print(f"Null counts (out of {total_rows:,} rows):")
for col_name, n_nulls in null_counts_row.items():
    pct = 100 * n_nulls / total_rows
    print(f"  {col_name:<25} {n_nulls:>8,}  ({pct:5.2f}%)")

Null counts (out of 393,890 rows):
  Id                               0  ( 0.00%)
  ProductId                        0  ( 0.00%)
  UserId                           0  ( 0.00%)
  ProfileName                      0  ( 0.00%)
  HelpfulnessNumerator             0  ( 0.00%)
  HelpfulnessDenominator           0  ( 0.00%)
  Score                            0  ( 0.00%)
  Time                             0  ( 0.00%)
  Summary                          0  ( 0.00%)
  Text                             0  ( 0.00%)
  ReviewDate                       0  ( 0.00%)
  Year                             0  ( 0.00%)
  Month                            0  ( 0.00%)
  HelpfulnessRatio           184,612  (46.87%)


**What we expect**: `HelpfulnessRatio` will have a lot of nulls — that's by design
(it's null whenever no one voted). `Summary` and `Text` should have very few. We drop
any rows missing the actual review `Text` since that's our main signal.

In [17]:
# Drop rows with missing Text — without text there's nothing to analyse
reviews_clean_df = reviews_clean_df.filter(F.col("Text").isNotNull())
print(f"Rows with non-null Text: {reviews_clean_df.count():,}")

Rows with non-null Text: 393,890


### 4.3 Derived features

Two simple extras for use later in EDA and ML:

- `TextLength` — character count
- `WordCount` — approximate token count (whitespace split)

For the ML pipeline we'll use a proper Tokenizer, but for EDA this is enough.

In [18]:
reviews_clean_df = (
    reviews_clean_df
        .withColumn("TextLength", F.length(F.col("Text")))
        # split() returns an array, size() returns its length — both native Spark
        .withColumn("WordCount",  F.size(F.split(F.col("Text"), r"\s+")))
)

# Cache — we'll run many EDA queries against this
reviews_clean_df.cache()

# Materialise the cache and report the final shape
final_rows = reviews_clean_df.count()
print(f"Cleaned dataset: {final_rows:,} rows, {len(reviews_clean_df.columns)} columns")
reviews_clean_df.printSchema()

Cleaned dataset: 393,890 rows, 16 columns
root
 |-- Id: integer (nullable = true)
 |-- ProductId: string (nullable = true)
 |-- UserId: string (nullable = true)
 |-- ProfileName: string (nullable = true)
 |-- HelpfulnessNumerator: integer (nullable = true)
 |-- HelpfulnessDenominator: integer (nullable = true)
 |-- Score: integer (nullable = true)
 |-- Time: long (nullable = true)
 |-- Summary: string (nullable = true)
 |-- Text: string (nullable = true)
 |-- ReviewDate: timestamp (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- HelpfulnessRatio: double (nullable = true)
 |-- TextLength: integer (nullable = true)
 |-- WordCount: integer (nullable = true)



In [19]:
# Quick preview
reviews_clean_df.select(
    "Id", "ProductId", "UserId", "Score", "HelpfulnessRatio",
    "ReviewDate", "Year", "TextLength", "WordCount", "Summary"
).show(5, truncate=40)

+------+----------+------------------+-----+------------------+-------------------+----+----------+---------+----------------------------------------+
|    Id| ProductId|            UserId|Score|  HelpfulnessRatio|         ReviewDate|Year|TextLength|WordCount|                                 Summary|
+------+----------+------------------+-----+------------------+-------------------+----+----------+---------+----------------------------------------+
| 83318|B005ZBZLT4|#oc-R115TNMSPFT9I7|    2|0.6666666666666666|2012-03-12 00:00:00|2012|       719|      132|"Green" K-cup packaging sacrifices fl...|
|136346|B006Q820X0|#oc-R120LO6OLNDPCG|    1|               0.5|2012-05-29 00:00:00|2012|       439|       86|                    Don't purchase this.|
|516181|B008I1XPKA|#oc-R12N3533IO3B79|    2|               0.6|2012-09-06 00:00:00|2012|       203|       38|                                   kcups|
|516204|B008I1XPKA|#oc-R19B7LHEK1ARMD|    5|               0.0|2012-10-18 00:00:00|2012|      

---
## 5. Exploratory Data Analysis

We explore the cleaned data using both the DataFrame API and Spark SQL. Performance
is identical between them — they compile to the same internal plan. We mix them
because the brief asks for both, and some queries are just clearer in SQL.

First, register the DataFrame as a SQL view so we can query it with `spark.sql(...)`.

In [20]:
reviews_clean_df.createOrReplaceTempView("reviews")

# Cache the SQL view too
spark.sql("CACHE TABLE reviews")
print("Registered temp view 'reviews' and cached it.")

Registered temp view 'reviews' and cached it.


### 5.1 Headline stats

In [21]:
spark.sql("""
    SELECT
        COUNT(*)                              AS total_reviews,
        COUNT(DISTINCT UserId)                AS unique_users,
        COUNT(DISTINCT ProductId)             AS unique_products,
        MIN(ReviewDate)                       AS earliest_review,
        MAX(ReviewDate)                       AS latest_review,
        ROUND(AVG(Score), 3)                  AS avg_score,
        ROUND(AVG(TextLength), 1)             AS avg_text_length,
        ROUND(AVG(WordCount), 1)              AS avg_word_count
    FROM reviews
""").show(truncate=False)

+-------------+------------+---------------+-------------------+-------------------+---------+---------------+--------------+
|total_reviews|unique_users|unique_products|earliest_review    |latest_review      |avg_score|avg_text_length|avg_word_count|
+-------------+------------+---------------+-------------------+-------------------+---------+---------------+--------------+
|393890       |256059      |67612          |1999-10-08 00:00:00|2012-10-26 00:00:00|4.179    |434.3          |79.8          |
+-------------+------------+---------------+-------------------+-------------------+---------+---------------+--------------+



### 5.2 Score distribution

How are the 1-5 star ratings distributed? This matters for ML — if the dataset is
heavily skewed toward 5-star reviews (which we expect), we'll need to handle the
imbalance.

In [22]:
score_dist = spark.sql("""
    SELECT
        Score,
        COUNT(*)                              AS n_reviews,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_total
    FROM reviews
    GROUP BY Score
    ORDER BY Score
""")

score_dist.show()

+-----+---------+------------+
|Score|n_reviews|pct_of_total|
+-----+---------+------------+
|    1|    36302|        9.22|
|    2|    20801|        5.28|
|    3|    29768|        7.56|
|    4|    56086|       14.24|
|    5|   250933|       63.71|
+-----+---------+------------+



In [23]:
# Spark's built-in .plot aggregates BEFORE plotting, so the driver only sees the
# 5-row result — safe. Never call .toPandas() on the full reviews DataFrame.
fig = score_dist.plot.bar(
    x="Score", y="n_reviews",
    title="Review-score distribution (1-5 stars)"
)
fig.show()

**Observation**: as expected, the distribution leans heavily toward 5-star reviews
(~63%), with 1-star a distant second. For binary sentiment classification we'll collapse
`[1,2] -> negative` and `[4,5] -> positive`, dropping 3-star reviews. The imbalance will
need handling in the ML notebook.

### 5.3 Review volume over time

In [24]:
yearly_volume = spark.sql("""
    SELECT
        Year,
        COUNT(*)              AS n_reviews,
        ROUND(AVG(Score), 3)  AS avg_score
    FROM reviews
    WHERE Year IS NOT NULL
    GROUP BY Year
    ORDER BY Year
""")

yearly_volume.show()

+----+---------+---------+
|Year|n_reviews|avg_score|
+----+---------+---------+
|1999|        4|      5.0|
|2000|       17|    4.706|
|2001|        8|     3.75|
|2002|       33|    4.788|
|2003|       99|    4.444|
|2004|      445|    4.535|
|2005|     1069|    4.497|
|2006|     4748|    4.298|
|2007|    15778|    4.387|
|2008|    23129|    4.344|
|2009|    37891|    4.281|
|2010|    58003|    4.214|
|2011|   113487|    4.138|
|2012|   139179|    4.112|
+----+---------+---------+



In [25]:
fig = yearly_volume.plot.area(
    x="Year", y="n_reviews",
    title="Number of reviews per year"
)
fig.show()

### 5.4 Most reviewed products and most active users

A classic power-law question. Using the DataFrame API this time (rather than SQL) to
show both styles.

Two things to notice:
- We use `limit(10)` before showing — never sort an entire DataFrame without a limit.
- We sort *after* grouping, so Spark only sorts the aggregated result (one row per
  product), not the full review dataset.

In [26]:
# Top 10 products by review count
top_products = (
    reviews_clean_df
        .groupBy("ProductId")
        .agg(
            F.count("*").alias("n_reviews"),
            F.round(F.avg("Score"), 3).alias("avg_score"),
            F.round(F.avg("HelpfulnessRatio"), 3).alias("avg_helpfulness")
        )
        .orderBy(F.desc("n_reviews"))
        .limit(10)
)

top_products.show(truncate=False)

+----------+---------+---------+---------------+
|ProductId |n_reviews|avg_score|avg_helpfulness|
+----------+---------+---------+---------------+
|B007JFMH8M|912      |4.582    |0.901          |
|B002QWP89S|630      |4.594    |0.784          |
|B003B3OOPA|622      |4.741    |0.878          |
|B001EO5Q64|566      |4.746    |0.86           |
|B0013NUGDE|558      |4.31     |0.738          |
|B000KV61FC|556      |3.412    |0.745          |
|B000UBD88A|542      |4.343    |0.791          |
|B000NMJWZO|542      |4.882    |0.906          |
|B005K4Q37A|541      |3.817    |0.655          |
|B0090X8IPM|530      |3.813    |0.486          |
+----------+---------+---------+---------------+



In [27]:
# Top 10 most active users
top_users = (
    reviews_clean_df
        .groupBy("UserId")
        .agg(
            F.count("*").alias("n_reviews"),
            F.round(F.avg("Score"), 3).alias("avg_score"),
            F.round(F.avg("TextLength"), 1).alias("avg_text_length")
        )
        .orderBy(F.desc("n_reviews"))
        .limit(10)
)

top_users.show(truncate=False)

+--------------+---------+---------+---------------+
|UserId        |n_reviews|avg_score|avg_text_length|
+--------------+---------+---------+---------------+
|AY12DBB0U420B |329      |4.66     |820.6          |
|A3OXHLG6DIBRW8|278      |4.547    |611.2          |
|A281NPSIMI1C2R|259      |4.788    |1004.7         |
|A1YUL9PCJR3JTY|214      |4.621    |1409.5         |
|A1Z54EM24Y40LL|211      |4.384    |592.3          |
|A2MUGFV2TDQ47K|161      |3.826    |611.9          |
|A3D6OI36USYOU1|146      |4.418    |853.6          |
|AZV26LP92E6WU |129      |4.853    |301.7          |
|AKMEY1BSHSDG7 |119      |4.748    |469.3          |
|A2GEZJHBV92EVR|118      |4.542    |282.5          |
+--------------+---------+---------+---------------+



### 5.5 Does review length correlate with helpfulness?

Reasonable hypothesis: longer reviews are more thoughtful and get rated as more helpful.
We bucket reviews by length and look at the average helpfulness per bucket.

In [28]:
length_vs_help = spark.sql("""
    SELECT
        CASE
            WHEN WordCount <  20             THEN '1. Very short (<20 words)'
            WHEN WordCount <  50             THEN '2. Short (20-49)'
            WHEN WordCount < 100             THEN '3. Medium (50-99)'
            WHEN WordCount < 200             THEN '4. Long (100-199)'
            ELSE                                  '5. Very long (200+)'
        END                                  AS length_bucket,
        COUNT(*)                             AS n_reviews,
        ROUND(AVG(Score), 3)                 AS avg_score,
        ROUND(AVG(HelpfulnessRatio), 3)      AS avg_helpfulness_ratio,
        -- Only counting reviews where someone actually voted on helpfulness
        ROUND(AVG(CASE WHEN HelpfulnessDenominator > 0
                       THEN HelpfulnessRatio END), 3) AS avg_helpfulness_when_voted
    FROM reviews
    GROUP BY length_bucket
    ORDER BY length_bucket
""")

length_vs_help.show(truncate=False)

+-------------------------+---------+---------+---------------------+--------------------------+
|length_bucket            |n_reviews|avg_score|avg_helpfulness_ratio|avg_helpfulness_when_voted|
+-------------------------+---------+---------+---------------------+--------------------------+
|1. Very short (<20 words)|11438    |4.464    |0.712                |0.712                     |
|2. Short (20-49)         |159249   |4.292    |0.762                |0.762                     |
|3. Medium (50-99)        |127642   |4.129    |0.799                |0.799                     |
|4. Long (100-199)        |71776    |4.035    |0.811                |0.811                     |
|5. Very long (200+)      |23785    |3.999    |0.813                |0.813                     |
+-------------------------+---------+---------+---------------------+--------------------------+



**Observation**: very short reviews get little engagement; helpfulness tends to peak
on medium-to-long reviews. This is real signal we can use in the helpfulness model in
the next notebook.

### 5.6 Window functions — finding each product's top reviewer

A nice example of window functions. For each product, we rank its reviewers by number
of reviews and keep the top one. This kind of "for each X, find the top Y" pattern is
extremely common in real consulting work.

In [29]:
top_reviewer_per_product = spark.sql("""
    WITH user_product_counts AS (
        SELECT
            ProductId,
            UserId,
            COUNT(*) AS n_reviews_by_this_user
        FROM reviews
        GROUP BY ProductId, UserId
    ),
    ranked AS (
        SELECT
            ProductId,
            UserId,
            n_reviews_by_this_user,
            ROW_NUMBER() OVER (PARTITION BY ProductId
                               ORDER BY n_reviews_by_this_user DESC) AS rnk
        FROM user_product_counts
    )
    SELECT ProductId, UserId, n_reviews_by_this_user
    FROM ranked
    WHERE rnk = 1
      AND n_reviews_by_this_user >= 5     -- only products with a real "super-reviewer"
    ORDER BY n_reviews_by_this_user DESC
    LIMIT 10
""")

top_reviewer_per_product.show(truncate=False)

+----------+--------------+----------------------+
|ProductId |UserId        |n_reviews_by_this_user|
+----------+--------------+----------------------+
|B000FYYOYO|AKJOL2Y2XJT00 |6                     |
|B003D4F1QS|A13HRSMJ5TOWEZ|6                     |
|B000GINU0S|A3FY3H6F4249E0|6                     |
|B0018KR8V0|A1P2XYD265YE21|6                     |
|B000WFUL3E|A29JUMRL1US6YP|5                     |
|B001EQ4P2I|A1X1CEGHTHMBL1|5                     |
|B0013NUGDE|A1LTNRGWZFY4C9|5                     |
|B001TLY7A8|A3D0J18LY62I06|5                     |
|B0034KP00S|A1TMAVN4CEM8U8|5                     |
|B003VXFK44|A2SZLNSI5KOQJT|5                     |
+----------+--------------+----------------------+



### 5.7 Sentiment label preview

Looking ahead to the ML notebook, we'll collapse scores into a binary sentiment label.
Worth checking the distribution now.

In [30]:
sentiment_preview = spark.sql("""
    SELECT
        CASE
            WHEN Score <= 2 THEN 'negative'
            WHEN Score >= 4 THEN 'positive'
            ELSE                 'neutral'
        END AS sentiment,
        COUNT(*) AS n_reviews,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM reviews
    GROUP BY sentiment
    ORDER BY n_reviews DESC
""")

sentiment_preview.show()

+---------+---------+-----+
|sentiment|n_reviews|  pct|
+---------+---------+-----+
| positive|   307019|77.95|
| negative|    57103|14.50|
|  neutral|    29768| 7.56|
+---------+---------+-----+



The imbalance is severe (~78% positive vs ~14% negative once neutrals are dropped).
We'll handle this with class weights in the ML notebook.

---
## 6. Saving the Cleaned Dataset

We save the cleaned DataFrame as **Parquet** for the downstream notebooks. Parquet is
the standard big-data file format:

- Columnar (queries only read the columns they need)
- Compressed (typically 5-10x smaller than CSV)
- Embeds the schema (no inference needed when re-reading)
- Splittable (multiple workers can read it in parallel)

We partition by `Year` — this means a query like `WHERE Year = 2010` only touches the
2010 folder, skipping everything else. Big speedup at scale.

In [31]:
OUTPUT_PATH = f"{PROJECT_DIR}/reviews_cleaned.parquet"

(
    reviews_clean_df
        .write
        .mode("overwrite")
        .partitionBy("Year")
        .parquet(OUTPUT_PATH)
)

print(f"Wrote cleaned data to {OUTPUT_PATH} (partitioned by Year).")

# Sanity check — re-read it
check_df = spark.read.parquet(OUTPUT_PATH)
print(f"Re-read row count: {check_df.count():,}")
print(f"Partitions detected on disk: {check_df.rdd.getNumPartitions()}")

Wrote cleaned data to /content/drive/MyDrive/Colab Notebooks/reviews_cleaned.parquet (partitioned by Year).
Re-read row count: 393,890
Partitions detected on disk: 7


---
## 7. Wrap-up

In this notebook we:

1. Set up Spark with sensible config.
2. Loaded the raw data as an RDD (showing `textFile`, `filter`, `map`, `flatMap`, `reduceByKey`, caching, `top()`).
3. Re-loaded with the DataFrame API using an explicit schema (avoiding the inference pass).
4. Cleaned the data: timestamp conversion, safe helpfulness ratio, removed impossible rows, deduplicated, audited nulls, added derived features.
5. Explored the data with both DataFrame API and Spark SQL (including CTEs, window functions, and plotting).
6. Saved the result as partitioned Parquet for the next notebooks.

### Big-data-safe practices we followed
- Explicit schema instead of `inferSchema=True`
- `top(N)` and `limit(N)` instead of unbounded sorts
- `reduceByKey` instead of `groupByKey` on RDDs
- Native Spark functions (no Python UDFs)
- `.cache()` only on DataFrames we reuse, with `.unpersist()` afterwards
- No `.collect()` or `.toPandas()` on the full dataset — only on small pre-aggregated results

### Coming up
- **Notebook 2**: ML Pipelines for sentiment classification and helpfulness prediction
- **Notebook 3**: Deep learning (DistilBERT) for sentiment
- **Notebook 4**: GraphFrames analysis of the user-product review graph
- **Notebook 5**: Structured Streaming (bonus)

In [32]:
# Clean shutdown — release caches and stop the session
spark.sql("UNCACHE TABLE reviews")
reviews_clean_df.unpersist()
spark.stop()
print("Spark session stopped.")

Spark session stopped.
